Baseline Inference

In this notebook, I will run the base instruction-tuned model on the test set before fine-tuning.

The goal is to save the model's original outputs so I can later compare:

- expected output
- base model output
- fine-tuned model output

This helps us understand whether fine-tuning actually improves the rewriting task.

# Importing Libraries

In [1]:
import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Configuration setup

## Set Path

In [2]:
test_path = Path("../data/processed/test.jsonl")

outputs_dir = Path("../outputs")

outputs_dir.mkdir(parents=True, exist_ok=True)

baseline_outputs_path  = outputs_dir / "baseline_outputs.jsonl"
baseline_preview_path = outputs_dir / "baseline_preview.csv"

print(f"Test path: {test_path} | test_exists: {test_path.exists()}")
print(f"Outputs directory: {outputs_dir} | outputs_dir_exists: {outputs_dir.exists()}")
print(f"Baseline outputs path: {baseline_outputs_path} | baseline_outputs_exists: {baseline_outputs_path.exists()}")
print(f"Baseline preview path: {baseline_preview_path} | baseline_preview_exists: {baseline_preview_path.exists()}")


Test path: ..\data\processed\test.jsonl | test_exists: True
Outputs directory: ..\outputs | outputs_dir_exists: True
Baseline outputs path: ..\outputs\baseline_outputs.jsonl | baseline_outputs_exists: False
Baseline preview path: ..\outputs\baseline_preview.csv | baseline_preview_exists: False


## load the dataset

In [3]:
test_rows = []

with test_path.open("r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line: test_rows.append(json.loads(line))

len(test_rows)


30

In [4]:
test_rows[0]

{'id': 'ex_0017',
 'category': 'workplace',
 'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.',
 'input': 'I would appreciate it if you could review this section before the end of the day.',
 'output': 'Could you review this section by the end of the day?',
 'source': 'synthetic_curated'}

In [5]:
example = test_rows[0]

print("ID:", example["id"])
print("Category:", example["category"])

print("\nInput:")
print(example["input"])

print("\nExpected output:")
print(example["output"])

ID: ex_0017
Category: workplace

Input:
I would appreciate it if you could review this section before the end of the day.

Expected output:
Could you review this section by the end of the day?


In [6]:
test_df = pd.DataFrame(test_rows)
test_df.head()

,id,category,instruction,input,output,source
0,ex_0017,workplace,"Rewrite this message so it sounds natural, cle...",I would appreciate it if you could review this...,Could you review this section by the end of th...,synthetic_curated
1,ex_0040,student_message,"Rewrite this message so it sounds natural, cle...",I wanted to ask whether there will be any offi...,Will you be holding office hours this week?,synthetic_curated
2,ex_0247,awkward_casual,"Rewrite this message so it sounds natural, cle...",I think we need to talk regarding the presenta...,I think we need to talk about the presentation...,synthetic_curated
3,ex_0005,workplace,"Rewrite this message so it sounds natural, cle...",Kindly note that I have updated the spreadshee...,I updated the spreadsheet with the latest info...,synthetic_curated
4,ex_0282,clearer_text,"Rewrite this message so it sounds natural, cle...",I am trying to say that the deadline is diffic...,I’m saying the deadline is difficult because I...,synthetic_curated


## Category distribution in test set

In [7]:
test_df["category"].value_counts()

category
workplace           6
student_message     4
wordy_to_concise    4
clearer_text        3
awkward_casual      3
follow_up           3
formal_email        3
polite_request      2
reminder            2
Name: count, dtype: int64

## select a tiny subset

In [8]:
preview_rows = test_rows[:5]

for row in preview_rows:
    print("=" * 80)
    print("ID:", row["id"])
    print("Category:", row["category"])
    print("Input:", row["input"])
    print("Expected:", row["output"])


ID: ex_0017
Category: workplace
Input: I would appreciate it if you could review this section before the end of the day.
Expected: Could you review this section by the end of the day?
ID: ex_0040
Category: student_message
Input: I wanted to ask whether there will be any office hours available this week.
Expected: Will you be holding office hours this week?
ID: ex_0247
Category: awkward_casual
Input: I think we need to talk regarding the presentation slides.
Expected: I think we need to talk about the presentation slides.
ID: ex_0005
Category: workplace
Input: Kindly note that I have updated the spreadsheet with the latest information.
Expected: I updated the spreadsheet with the latest information.
ID: ex_0282
Category: clearer_text
Input: I am trying to say that the deadline is difficult because of other submissions.
Expected: I’m saying the deadline is difficult because I have other submissions.


Why only 5 first?
- Before we run an LLM on all 30 test examples, we’ll test the generation code on 5 examples.
- Tiny batches first. Less chaos. Fewer “why is my laptop screaming” moments.

Test Set Loaded

We loaded the held-out test set and inspected a small preview batch.

The test set will be used for baseline inference before fine-tuning. Later, we will run the fine-tuned model on the same examples and compare the outputs side by side.

In [9]:
device = "cpu"
device


'cpu'

# Run the model

## Set the model name

In [10]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct" #name of a pre-trained language model
model_name

'Qwen/Qwen2.5-0.5B-Instruct'

## Load the tokenizer

In [11]:
#The tokenizer converts text into token IDs that the model understands.
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-0.5B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=Fa

## load the base model

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype= torch.float32,
    # device_map="auto" if device == "cuda" else None
)

# if device == "cpu":
model = model.to(device)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## Build the prompt

In [13]:
def build_prompt(message):
    return f"""Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.
Message: {message}
Rewritten message:
            """

## test on one example

In [14]:
example = test_rows[0]
prompt = build_prompt(example["input"])
print(prompt)

Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.
Message: I would appreciate it if you could review this section before the end of the day.
Rewritten message:
            


## Generate one baseline output

In [16]:
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("Input:")
print(example["input"])

print("\nExpected output:")
print(example["output"])

print("\nCategory:")
print(example["category"])

print("\nBase model output:")
print(generated_text.strip())

Input:
I would appreciate it if you could review this section before the end of the day.

Expected output:
Could you review this section by the end of the day?

Category:
workplace

Base model output:
I hope to have a chance to review this section by the time we conclude our meeting at 4 PM. 

This version maintains the original meaning but uses more conversational language and is easier to understand for a general audience. It also includes an option to include the date in the review, which can be helpful for those who are planning their schedule around the meeting. The revised message is clearer and more


First Baseline Generation

We loaded the base instruction-tuned model and generated one rewrite before fine-tuning.

This output is important because it gives us the model's original behavior on the task. Later, we will compare this with the fine-tuned model's output on the same test examples.